# Python for Health, Economic, and Social Science
## Lab 2 — Instructor Solution & Expansion Notes

**Course:** LCDS, Nuffield College Oxford  
**Day 2 theme:** For loops, error handling, functions, and file I/O

---

### Instructor framing for Day 2

Day 2 is where Python starts feeling useful for data work. The progression:
- **Loops** → iterate over all rows in a dataset, not just one
- **Comprehensions** → build cleaned/filtered lists in one readable expression
- **Error handling** → write code that survives messy real-world data
- **Sets** → deduplicate and find overlaps between indicator lists or cohort IDs

All four skills are used daily in ONS/NHS/World Bank data processing pipelines.

---
## Part A: Strings, Sets, and Basic Text Processing

### Q1: Word count with `split()` and `len()`

In [2]:
# Answer 1
policy_statement = 'Integrated health social and economic policy improves long term wellbeing outcomes.'

word_count = len(policy_statement.split())
print(f"Word count: {word_count}")

Word count: 11



> **Expansion — word count in policy text analysis:**
```python
texts = [
    "Integrated health social and economic policy improves long term wellbeing outcomes.",
    "School absence is closely linked to poverty and housing instability.",
    "Preventive care reduces emergency admissions in high-deprivation areas."
]
for i, t in enumerate(texts, start=1):
    words = t.split()
    print(f"Text {i}: {len(words)} words, avg word length: {sum(len(w) for w in words)/len(words):.1f}")
```
> Average word length and sentence length are simple readability proxies used in public health communication research.

### Q2: Unique letters in a report line

In [3]:
# Answer 2
report_line1 = "Clinic visits are rising in Northside"
report_line2 = "School absence is falling in Central"

# Set of unique letters; lower() unifies case, filter keeps only letters
letters1 = set(ch for ch in report_line1.lower() if ch.isalpha())
print(f"Unique letters in report_line1: {len(letters1)}")

Unique letters in report_line1: 14



>
> **Expansion — sets for cohort deduplication:**
```python
# In administrative health data, the same patient ID may appear multiple times.
# Sets give you unique IDs instantly:
patient_ids_clinic   = [1001, 1002, 1003, 1001, 1002, 1005]
patient_ids_hospital = [1002, 1004, 1005, 1006]

unique_clinic   = set(patient_ids_clinic)
unique_hospital = set(patient_ids_hospital)

print(f"Unique clinic patients: {len(unique_clinic)}")
print(f"Seen in both (linkage): {unique_clinic & unique_hospital}")
print(f"Clinic only (not hospitalised): {unique_clinic - unique_hospital}")
```
> This is the logic behind record linkage — a core workflow in health economics research.

### Q3: Overlapping letter sets

In [ ]:
# Answer 3
report_line1 = "Clinic visits are rising in Northside"
report_line2 = "School absence is falling in Central"

letters1 = set(ch for ch in report_line1.lower() if ch.isalpha())
letters2 = set(ch for ch in report_line2.lower() if ch.isalpha())

overlap = letters1 & letters2     # intersection
print(f"Shared letters: {sorted(overlap)}")

### Q4: Phrase repair with `split` / `replace` / `join`

In [ ]:
# Answer 4
wrongphrase = "Northside low clinic visits and high school attendance"
words = wrongphrase.split()

# Swap the adjectives: positions 1 and 5
words[1], words[5] = words[5], words[1]

print(" ".join(words))

>  The swap `a, b = b, a` is elegant Python. It is also a useful micro-example of tuple unpacking — which becomes more important when working with `enumerate()`, `zip()`, and pandas `.iterrows()`.
>
> **Expansion — targeted word replacement without index arithmetic:**
```python
wrongphrase = "Northside low clinic visits and high school attendance"
# When you know the *words* to swap (not the positions):
corrected = wrongphrase.replace("low clinic visits and high",
                                "high clinic visits and low")
print(corrected)
```
> `.replace()` is fine for known patterns; for flexible pattern matching (e.g., 'any digit followed by %') use `re.sub()` — covered in the Day 5 NLP content.

### Q5: Loop with condition on first letter

In [ ]:
# Answer 5
analysis_text = "Urban unemployment usually increases uncertainty unless urgent support is available"

for word in analysis_text.split():
    if word[0].lower() == 'u':
        print(word)

> **Expansion — finding domain keywords in policy text:**
```python
# Real use: flag sentences containing risk-related keywords
risk_keywords = {"unemployment", "poverty", "absence", "emergency", "arrears", "sanction"}
sentences = [
    "Unemployment rates are high in Riverside.",
    "Preventive care visits increased this quarter.",
    "Families face housing arrears and poverty."
]
for sent in sentences:
    words_lower = set(sent.lower().split())
    flags = words_lower & risk_keywords
    if flags:
        print(f"[FLAGGED] '{sent}' | triggers: {flags}")
```
> On Day 5 (NLP), students will build a proper keyword scorer with weights. This is the conceptual precursor.

### Q6: Dictionary replacement

In [4]:
# Answer 6
a_descriptive_phrase = "My area is Northside and my role is analyst."
about_you = {"Northside": "your area", "analyst": "your role"}

words = a_descriptive_phrase.split()
replaced = [about_you.get(w, about_you.get(w.rstrip('.'), w + '.' if w.endswith('.') else w))
            for w in words]

# Cleaner approach: strip punctuation, replace, re-attach
output_words = []
for w in words:
    punct = w[-1] if not w[-1].isalpha() else ''
    core = w.rstrip(punct) if punct else w
    replacement = about_you.get(core, core)
    output_words.append(replacement + punct)

print(" ".join(output_words))

My area is your area and my role is your role.


In [7]:
replaced = [x if x not in about_you.keys() else about_you[x] for x in a_descriptive_phrase.split()]
print(" ".join(output_words))

My area is your area and my role is your role.


> **Instructor note:** Trailing punctuation attached to words is a classic pain point in text processing. Show this bug explicitly — `about_you.get('analyst.')` returns `None` because the key is `'analyst'` not `'analyst.'`. The strip-then-reattach pattern is robust.
>
> **Simpler reference solution (acceptable for students):**
```python
phrase = a_descriptive_phrase
for old, new in about_you.items():
    phrase = phrase.replace(old, new)
print(phrase)
```

---
## Part B: Loops, Comprehensions, and Conditions

### Q7: For loop with if/elif conditions on risk score

In [8]:
# Answer 7
import random
random.seed(42)  # not required by question but helps reproducibility

for i in range(10):
    risk_score = random.randint(0, 9)
    if risk_score % 2 != 0:               # odd
        risk_score *= 2
        print(f"  WARNING: odd score detected — adjusted to {risk_score}")
    if risk_score > 4:
        print(f"  REVIEW NEEDED: score {risk_score} — assess health/social/economic risk")

  REVIEW NEEDED: score 6 — assess health/social/economic risk
  REVIEW NEEDED: score 6 — assess health/social/economic risk
  REVIEW NEEDED: score 8 — assess health/social/economic risk
  REVIEW NEEDED: score 18 — assess health/social/economic risk


> Two *separate* if statements (not if/elif) — the second check applies to the (potentially updated) value. Draw the control-flow diagram on the board: two independent gates, not a branch. This distinction matters hugely in risk-flagging pipelines.
>
> **Expansion — audit trail with `enumerate`:**
```python
import random
random.seed(42)
log = []

for i in range(10):
    raw_score = random.randint(0, 9)
    adjusted = raw_score * 2 if raw_score % 2 != 0 else raw_score
    flag = adjusted > 4
    log.append({"iteration": i, "raw": raw_score, "adjusted": adjusted, "flagged": flag})

flagged_only = [r for r in log if r["flagged"]]
print(f"{len(flagged_only)} / {len(log)} iterations flagged for review")
```
> Appending to an audit log list is the precursor to building a DataFrame of results — shown on Day 4.

### Q8: Unique area codes

In [ ]:
# Answer 8
import random
random.seed(99)

seen = set()
for i in range(10):
    area_code = random.randint(0, 9)
    if area_code not in seen:
        print(area_code)
        seen.add(area_code)

>  Using a `set` for the `seen` tracker is O(1) lookup. Using a list would be O(n) per iteration — an important difference when processing millions of administrative records.
>
> **Expansion — first-occurrence deduplication on a LSOA code list:**
```python
# Typical task: process each LSOA (Lower Super Output Area) once from a long repeated
# list of GP registration records.
lsoa_stream = ["E01000001", "E01000002", "E01000001", "E01000003", "E01000002"]

seen_lsoas = set()
first_seen = []
for code in lsoa_stream:
    if code not in seen_lsoas:
        first_seen.append(code)
        seen_lsoas.add(code)

print("Unique LSOAs in order of first appearance:", first_seen)
# Note: dict.fromkeys() preserves insertion order in Python 3.7+:
print(list(dict.fromkeys(lsoa_stream)))
```

### Q9: List comprehension — risk index

In [ ]:
# Answer 9
risk_index = [i**i for i in range(0, 10)]
print(risk_index)

> `0**0 = 1` by Python convention — students occasionally query this. Explain it is a mathematical convention (limit definition) rather than a bug. The broader point: comprehensions are more readable than equivalent loops for transformations and should be the default when there is no side-effect required.
>
> **Expansion — comprehensions for demographic rate transforms:**
```python
# Transform raw counts to per-1,000 rates in one expression:
clinic_visits = [120, 95, 140, 102, 110]
populations   = [2400, 1800, 2600, 2100, 2000]

rates_per_1000 = [round(v/p * 1000, 1) for v, p in zip(clinic_visits, populations)]
print(rates_per_1000)

# With condition — flag high-rate areas only:
high_demand = [(v, p) for v, p in zip(clinic_visits, populations) if v/p * 1000 > 50]
print("High-demand areas (visits, pop):", high_demand)
```

### Q10: List comprehension with `if` — extracting @-mentions

In [ ]:
# Answer 10
Update = "New analysis by @HealthUnit @CityDataLab and @PolicyGroup links income shocks to school absence"

AtMentions = [i for i in Update.split() if i[0] == '@']
print(AtMentions)

>  The `if` clause filters *before* appending — it is not a separate `filter()` call on the result. This is syntactically different from an `if/else` expression, which transforms values (`x if cond else y`) vs. an `if` guard, which filters (`x for x in ... if cond`).
>
> **Expansion — extracting citations, hashtags, or URLs from social/media text:**
```python
post = "#PublicHealth data shows #inequality gap — see report at https://example.org/report"
hashtags = [w for w in post.split() if w.startswith('#')]
urls      = [w for w in post.split() if w.startswith('http')]
print("Hashtags:", hashtags)
print("URLs:", urls)
```
> On Day 5 (NLP), students will extract similar features from clinical case notes and score them for risk.

---
## Part C: Dictionaries and Error Handling

### Q11: Set operations on indicator lists

In [ ]:
# Answer 11
health_indicators    = {"clinic_visits", "hospital_wait_time", "vaccination_rate"}
economic_indicators  = {"median_income",  "unemployment_rate",  "vaccination_rate"}

print("Union:",        health_indicators | economic_indicators)   # 1.
print("Intersection:", health_indicators & economic_indicators)   # 2.
print("Health only:",  health_indicators - economic_indicators)   # 3.

> `vaccination_rate` appears in both — this is intentional. Vaccination is a classic bridging indicator used in both health outcome models (Lecture 5) and economic productivity models. The intersection (`&`) finds exactly these cross-domain indicators.
>
> **Expansion — tracking indicator availability across datasets:**
```python
# Which indicators are in your dataset but not the published standard set?
your_dataset_cols  = {"area", "clinic_visits", "gp_list_size", "deprivation_score"}
standard_framework = {"area", "clinic_visits", "hospital_wait_time", "life_expectancy"}

in_both    = your_dataset_cols & standard_framework
yours_only = your_dataset_cols - standard_framework
missing    = standard_framework - your_dataset_cols

print(f"Covered indicators: {in_both}")
print(f"Extra in your data: {yours_only}")
print(f"Missing from framework: {missing}")
```

### Q12: Dictionary iteration and income classification

In [ ]:
# Answer 12
area_income = {"Northside": 29000, "Central": 36000, "Riverside": 25000}

for area, income in area_income.items():
    label = "high income" if income >= 30000 else "lower income"
    print(f"{area}: {label} (£{income:,})")

>  `.items()` returns `(key, value)` tuples — the most common dict iteration pattern. Students who use `for key in dict` then index `dict[key]` inside the loop will work, but it is less readable. The ternary `value if cond else value2` is compact but loses clarity beyond two branches — for more levels, use `if/elif/else`.
>
> **Expansion — median-based classification (no hard-coded threshold):**
```python
incomes = list(area_income.values())
incomes_sorted = sorted(incomes)
n = len(incomes_sorted)
# Median for odd n:
median_income = incomes_sorted[n // 2]

for area, income in area_income.items():
    label = "above median" if income >= median_income else "below median"
    print(f"{area}: £{income:,} — {label}")
```
> Using the *sample* median rather than a fixed threshold is more defensible in policy analysis — the threshold adapts to the distribution in your data.

### Q13: Nested dictionary checks

In [ ]:
# Answer 13
area_snapshot = {
    "Northside": {"clinic_visits": 120, "school_absence_rate": 0.07},
    "Central":   {"clinic_visits": 95,  "school_absence_rate": 0.05},
    "Riverside": {"clinic_visits": 140, "school_absence_rate": 0.09}
}

for area, data in area_snapshot.items():
    if data["clinic_visits"] > 130 or data["school_absence_rate"] > 0.08:
        print(f"WARNING: {area} — visits={data['clinic_visits']}, absence={data['school_absence_rate']:.0%}")

> **Expansion — recording which threshold was breached:**
```python
for area, data in area_snapshot.items():
    triggers = []
    if data["clinic_visits"] > 130:
        triggers.append(f"high visits ({data['clinic_visits']})")
    if data["school_absence_rate"] > 0.08:
        triggers.append(f"high absence ({data['school_absence_rate']:.0%})")
    if triggers:
        print(f"{area}: ALERT — {', '.join(triggers)}")

# This builds an audit trail and is more informative for a policy briefing than a simple WARNING.
```

### Q14: List comprehension with dictionaries

In [ ]:
# Answer 14
records = [
    {"area": "Northside", "median_income": 29000},
    {"area": "Central",   "median_income": 36000},
    {"area": "Riverside", "median_income": 25000}
]

lower_income_areas = [r["area"] for r in records if r["median_income"] < 30000]
print(lower_income_areas)

> **Expansion — extracting both area and value together:**
```python
# Returns (area, income) tuples for lower-income areas:
lower_income_pairs = [(r["area"], r["median_income"]) for r in records if r["median_income"] < 30000]
for area, inc in lower_income_pairs:
    print(f"{area}: £{inc:,}")

# Dictionary comprehension — build a lookup dict of qualifying areas:
lower_income_lookup = {r["area"]: r["median_income"] for r in records if r["median_income"] < 30000}
print(lower_income_lookup)
```

### Q15: `try/except` for dirty data

In [ ]:
# Answer 15
raw_visits = ["120", "95", "missing", "140", "error", "102"]

clean_visits = []
for val in raw_visits:
    try:
        clean_visits.append(int(val))
    except ValueError:
        print(f"Skipping invalid entry: '{val}'")

print("\nClean visits:", clean_visits)

>`ValueError` is the correct exception class for failed type conversions. Catching bare `Exception` is a code smell — always narrow the exception type. Demonstrate this by trying `int(None)` (raises `TypeError`, not `ValueError`) to make the point concrete.
>
> **Expansion — full cleaning log with invalid entries tracked:**
```python
raw_visits = ["120", "95", "missing", "140", "n/a", "102", "9.5"]
clean_visits  = []
invalid_entries = []

for val in raw_visits:
    try:
        clean_visits.append(int(val))    # int('9.5') raises ValueError
    except ValueError:
        try:
            clean_visits.append(round(float(val)))  # handle '9.5' → 10
        except ValueError:
            invalid_entries.append(val)
            print(f"Cannot parse: '{val}'")

print(f"\nValid:   {clean_visits}")
print(f"Invalid: {invalid_entries}")
print(f"Data quality rate: {len(clean_visits)/len(raw_visits):.0%}")
```
> Data quality rate reporting is expected in NHS digital submissions and many social survey data notes.

---
## Instructor Expansion: Putting It All Together

### Mini pipeline: clean → classify → summarise

In [ ]:
# Full pipeline combining all Part C skills:
raw_data = [
    {"area": "Northside", "visits_str": "120", "income_str": "29000"},
    {"area": "Central",   "visits_str": "95",  "income_str": "36000"},
    {"area": "Riverside", "visits_str": "missing", "income_str": "25000"},
    {"area": "Eastbrook", "visits_str": "140", "income_str": "n/a"},
]

processed = []
for rec in raw_data:
    try:
        visits = int(rec["visits_str"])
        income = int(rec["income_str"])
        label  = "high demand" if visits > 110 else "normal demand"
        band   = "lower income" if income < 30000 else "higher income"
        processed.append({"area": rec["area"], "visits": visits, "income": income,
                           "demand_label": label, "income_band": band})
    except ValueError:
        print(f"  [SKIP] {rec['area']}: could not parse all fields")

print("\nProcessed records:")
for p in processed:
    print(f"  {p['area']}: {p['demand_label']}, {p['income_band']}")

> This combined loop is the **row-wise processing pattern** that pandas `.apply()` and vectorised operations replace on Day 4. Understanding it here makes the pandas API meaningful rather than magical.

---
*LCDS Oxford Python Course, March 2026 — Day 2 Lab Solution*